# PortfolioPilot - Milestone 3: cleaning, EDA, dimension reduction, feature set



**Run the cells in order, top to bottom.** Each cell uses variables made by the cells before it.
After a kernel restart, start again from cell 1.

Rule for M3: the test sets are created in Step 3 and then never explored. Every statistic, cap,
table and figure uses **training data only**.

In [3]:
%pip install numpy pandas scipy scikit-learn matplotlib yfinance

## Step 1 - Setting up

In [10]:
from google.colab import drive
drive.mount("/content/drive")          # a pop-up asks you to allow access: click Allow

import os
os.chdir("/content/drive/MyDrive/Portfolio_pilot/data/raw")
print("Working in:", os.getcwd())
print("Files here:", os.listdir())

from pathlib import Path
SCF_FILE = Path("SCFP2025.csv")
OUT_DIR  = Path("outputs")


SEED = 7275

# SCF column names in the summary extract (check against the SCF codebook)
COLS = {
    "household_id": "YY1", "record_id": "Y1", "weight": "WGT", "age": "AGE",
    "income": "INCOME", "networth": "NETWORTH", "liq": "LIQ", "debt": "DEBT",
    "equity": "EQUITY", "fin": "FIN", "yes_risk": "YESFINRISK", "no_risk": "NOFINRISK",
}
IMPLICATE = 1
SPLIT = {"train": 0.70, "val": 0.15, "test": 0.15}
CAP_QUANTILE = 0.99            # caps for ratio features, from training data

K_RANGE = [2, 3, 4, 5, 6]
CLUSTER_METHOD = "kmeans"      # "kmeans" or "gmm"
CHOSEN_K = None                # None = automatic (best silhouette); or set a number after Step 9
MIN_CLUSTER_SHARE = 0.05
N_BOOTSTRAP = 20               # use 3 for a quick test, 20 for final results
PCA_THRESHOLD = 0.80

TICKERS = ["SPY", "VEA", "VWO", "AGG", "TLT", "TIP", "VNQ", "GLD", "DBC"]
ETF_START, ETF_END = "2016-01-01", "2026-01-01"   # end is exclusive: data through 2025-12-31
ETF_TRAIN_END, ETF_VAL_END = "2021-12-31", "2023-12-31"
ETF_FILE = OUT_DIR / "raw" / "etf_prices_raw.csv"

FIG_DIR, TAB_DIR, DATA_DIR = OUT_DIR / "figures", OUT_DIR / "tables", OUT_DIR / "processed"
for d in [FIG_DIR, TAB_DIR, DATA_DIR, ETF_FILE.parent]:
    d.mkdir(parents=True, exist_ok=True)

print("SCF file found:", SCF_FILE.exists())
print("Outputs go to:", OUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working in: /content/drive/MyDrive/Portfolio_pilot/data/raw
Files here: ['SCFP2025.csv', 'PortfolioPilot_ETF_Prices.csv', 'outputs']
SCF file found: True
Outputs go to: outputs


## Step 2 - Import and helpers

In [13]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score

plt.rcParams.update({"figure.dpi": 110, "font.size": 9,
                     "axes.spines.top": False, "axes.spines.right": False})
np.random.seed(SEED)

NUMBERS = {}   # every count quoted in the report is stored here and saved in the last step
def note(section, key, value):
    NUMBERS.setdefault(section, {})[key] = value

def to_plain(v):
    """Make numpy values JSON-friendly."""
    if isinstance(v, dict):
        return {str(k): to_plain(x) for k, x in v.items()}
    if isinstance(v, (list, tuple)):
        return [to_plain(x) for x in v]
    if isinstance(v, np.integer):
        return int(v)
    if isinstance(v, (np.floating, float)):
        return round(float(v), 6)
    return v

def save_fig(fig, name):
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

def weighted_resample(df, seed):
    """Draw households with probability proportional to the survey weight."""
    rng = np.random.default_rng(seed)
    p = df["weight"].to_numpy(float)
    return df.iloc[rng.choice(len(df), size=len(df), replace=True, p=p / p.sum())]

def weighted_median(x, w):
    o = np.argsort(np.asarray(x))
    x, w = np.asarray(x, float)[o], np.asarray(w, float)[o]
    c = np.cumsum(w)
    return float(x[np.searchsorted(c, 0.5 * c[-1])])

print("Ready")

Ready


## Step 3 - Load the SCF and check its structure

In [14]:
raw = pd.read_csv(SCF_FILE, low_memory=False)
raw.columns = [c.upper() for c in raw.columns]
note("raw", "rows", len(raw))
note("raw", "columns", raw.shape[1])

missing_cols = [c for c in COLS.values() if c not in raw.columns]
assert not missing_cols, f"Columns not in the file: {missing_cols}. Fix COLS in Step 1."

full = raw[list(COLS.values())].copy()
full.columns = list(COLS.keys())
full["implicate"] = (full["record_id"] % 10).astype(int)    # Y1 = YY1*10 + implicate
note("raw", "households", full["household_id"].nunique())
note("raw", "implicates", sorted(full["implicate"].unique().tolist()))
del raw

print(NUMBERS["raw"])
full.head()

{'rows': 21800, 'columns': 362, 'households': 4360, 'implicates': [1, 2, 3, 4, 5]}


,household_id,record_id,weight,age,income,networth,liq,debt,equity,fin,yes_risk,no_risk,implicate
0,1,11,1241.701712,39,317351.35135,2434270.0,80300,250000.0,120150.0,215300,0,0,1
1,1,12,1247.038677,39,317351.35135,2208400.0,80300,250000.0,113800.0,229300,0,0,2
2,1,13,1234.428835,39,317351.35135,2128300.0,80300,250000.0,75300.0,209300,0,0,3
3,1,14,1267.530314,39,317351.35135,1742300.0,80300,250000.0,72800.0,212300,0,0,4
4,1,15,911.408367,39,317351.35135,1517300.0,80300,250000.0,73600.0,228300,0,0,5


## Step 4 - Cleaning: duplicates, missing values, impossible values

In [15]:
note("duplicates", "exact_duplicate_rows", int(full.duplicated().sum()))
note("duplicates", "duplicate_record_ids", int(full["record_id"].duplicated().sum()))
full = full.drop_duplicates(subset="record_id")

hh = full[full["implicate"] == IMPLICATE].copy()           # one row per household
note("implicate_1", "rows", len(hh))

na = hh.isna().sum()
note("missing", "by_column", na[na > 0].to_dict())
note("missing", "total", int(na.sum()))

checks = {
    "age_under_18_or_over_100": (hh["age"] < 18) | (hh["age"] > 100),
    "weight_not_positive": hh["weight"] <= 0,
    "liq_negative": hh["liq"] < 0,
    "debt_negative": hh["debt"] < 0,
    "equity_negative": hh["equity"] < 0,
    "fin_negative": hh["fin"] < 0,
    "equity_more_than_fin": hh["equity"] > hh["fin"] + 1,
    "liq_more_than_fin": hh["liq"] > hh["fin"] + 1,
    "both_risk_flags_1": (hh["yes_risk"] == 1) & (hh["no_risk"] == 1),
}
for name, mask in checks.items():
    note("errors", name, int(mask.sum()))
bad = checks["age_under_18_or_over_100"] | checks["weight_not_positive"] | checks["both_risk_flags_1"]
note("errors", "rows_dropped", int(bad.sum()))
hh = hh[~bad].reset_index(drop=True)

# Real values that need special handling later (not errors)
for name, mask in {"income_zero": hh["income"] == 0, "income_negative": hh["income"] < 0,
                   "networth_negative": hh["networth"] < 0, "fin_zero": hh["fin"] == 0,
                   "debt_zero": hh["debt"] == 0}.items():
    note("genuine", name, int(mask.sum()))

# Risk willingness: 0 = no risk, 1 = average, 2 = above-average / substantial
hh["risk_willingness"] = np.select([hh["no_risk"] == 1, hh["yes_risk"] == 1], [0, 2], default=1)
hh["age_band"] = pd.cut(hh["age"], [0, 34, 44, 54, 64, 74, 200],
                        labels=["<35", "35-44", "45-54", "55-64", "65-74", "75+"]).astype(str)
note("implicate_1", "rows_after_cleaning", len(hh))

for s in ["duplicates", "implicate_1", "missing", "errors", "genuine"]:
    print(s, NUMBERS[s])

duplicates {'exact_duplicate_rows': 0, 'duplicate_record_ids': 0}
implicate_1 {'rows': 4360, 'rows_after_cleaning': 4360}
missing {'by_column': {}, 'total': 0}
errors {'age_under_18_or_over_100': 0, 'weight_not_positive': 0, 'liq_negative': 0, 'debt_negative': 0, 'equity_negative': 0, 'fin_negative': 0, 'equity_more_than_fin': 0, 'liq_more_than_fin': 0, 'both_risk_flags_1': 0, 'rows_dropped': 0}
genuine {'income_zero': 23, 'income_negative': 0, 'networth_negative': 330, 'fin_zero': 53, 'debt_zero': 1115}


## Step 5 - Split train / validation / test (before any exploration)
Stratified on age band x risk willingness. Split by household, so implicates 2-5 can follow the same IDs later.


In [16]:
strata = hh["age_band"] + "|" + hh["risk_willingness"].astype(str)
counts = strata.value_counts()
strata = strata.where(strata.map(counts) >= 10, "other")      # merge tiny strata
note("split", "n_strata", strata.nunique())

train_val, test = train_test_split(hh, test_size=SPLIT["test"], stratify=strata, random_state=SEED)
train, val = train_test_split(train_val, test_size=SPLIT["val"] / (SPLIT["train"] + SPLIT["val"]),
                              stratify=strata.loc[train_val.index], random_state=SEED)
parts = {"train": train.reset_index(drop=True), "val": val.reset_index(drop=True),
         "test": test.reset_index(drop=True)}

for name, d in parts.items():
    note("split", f"{name}_households", len(d))
    note("split", f"{name}_weight_share", d["weight"].sum() / hh["weight"].sum())
pd.concat([pd.DataFrame({"household_id": d["household_id"], "partition": n}) for n, d in parts.items()]) \
  .to_csv(DATA_DIR / "scf_split_ids.csv", index=False)
NUMBERS["split"]

{'n_strata': 18,
 'train_households': 3052,
 'train_weight_share': np.float64(0.7033403995202065),
 'val_households': 654,
 'val_weight_share': np.float64(0.1441687614754587),
 'test_households': 654,
 'test_weight_share': np.float64(0.15249083900433474)}

## Step 6 - Implicate structure check

In [19]:
tr_full = full[full["household_id"].isin(parts["train"]["household_id"])]
g = tr_full.groupby("household_id")
note("implicates", "train_households_without_5_rows", int((g.size() != 5).sum()))
for c in ["income", "networth", "liq", "debt"]:
    note("implicates", f"share_where_{c}_varies", float((g[c].nunique() > 1).mean()))
NUMBERS["implicates"]

{'train_households_without_5_rows': 0,
 'share_where_income_varies': 0.6055045871559633,
 'share_where_networth_varies': 0.9121887287024901,
 'share_where_liq_varies': 0.4384010484927916,
 'share_where_debt_varies': 0.42660550458715596}

## Step 7 - Build features (caps fitted on training data only)

In [20]:
def ratio(num, den):
    return (num / den.where(den > 0)).astype(float)       # undefined (NaN) when den <= 0

tr = parts["train"]
CAPS = {"dti": float(ratio(tr["debt"], tr["income"]).quantile(CAP_QUANTILE)),
        "liq_to_income": float(ratio(tr["liq"], tr["income"]).quantile(CAP_QUANTILE))}
note("features", "caps_from_train", CAPS)
note("features", "train_ratio_undefined_income_le_0", int((tr["income"] <= 0).sum()))
note("features", "train_dti_above_cap", int((ratio(tr["debt"], tr["income"]) > CAPS["dti"]).sum()))
note("features", "train_lti_above_cap", int((ratio(tr["liq"], tr["income"]) > CAPS["liq_to_income"]).sum()))
note("features", "train_fin_zero", int((tr["fin"] <= 0).sum()))

def build_features(df):
    out = df.copy()
    nonpos = out["income"] <= 0
    out["income_nonpositive"] = nonpos.astype(int)
    dti = ratio(out["debt"], out["income"])
    dti = dti.where(~nonpos, np.where(out["debt"] > 0, CAPS["dti"], 0.0))
    out["dti"] = dti.clip(upper=CAPS["dti"])
    lti = ratio(out["liq"], out["income"])
    lti = lti.where(~nonpos, np.where(out["liq"] > 0, CAPS["liq_to_income"], 0.0))
    out["liq_to_income"] = lti.clip(upper=CAPS["liq_to_income"])
    out["equity_share"] = ratio(out["equity"], out["fin"]).fillna(0.0).clip(0, 1)
    for c in ["income", "networth", "liq", "dti", "liq_to_income"]:
        out[f"asinh_{c}"] = np.arcsinh(out[c].astype(float))
    return out

feats = {name: build_features(d) for name, d in parts.items()}

# Clustering variables define the target; classifier variables are the ones a user can give
CLUSTER_FEATURES = ["age", "risk_willingness", "asinh_networth", "asinh_dti",
                    "asinh_liq_to_income", "equity_share"]
CLASSIFIER_FEATURES = ["age", "risk_willingness", "asinh_income", "asinh_networth", "asinh_liq",
                       "asinh_liq_to_income", "income_nonpositive"]

assert not feats["train"][CLUSTER_FEATURES + CLASSIFIER_FEATURES].isna().any().any()
NUMBERS["features"]

{'caps_from_train': {'dti': 7.322105263125945,
  'liq_to_income': 4.421218403259946},
 'train_ratio_undefined_income_le_0': 18,
 'train_dti_above_cap': 31,
 'train_lti_above_cap': 31,
 'train_fin_zero': 34}

## Step 8 - Outlier check: 3×IQR fences on raw and inverse hyperbolic sine(asinh) transformed features (training data only)

In [21]:
rows = []
for col in ["income", "asinh_income", "networth", "asinh_networth", "liq", "asinh_liq"]:
    x = feats["train"][col].astype(float)
    q1, q3 = x.quantile([0.25, 0.75])
    n = int(((x < q1 - 3 * (q3 - q1)) | (x > q3 + 3 * (q3 - q1))).sum())
    rows.append({"variable": col, "n_beyond_3iqr": n, "min": x.min(), "max": x.max()})
outliers = pd.DataFrame(rows)
outliers.to_csv(TAB_DIR / "scf_outliers_train.csv", index=False)
note("outliers", "table", outliers.to_dict(orient="records"))
outliers

,variable,n_beyond_3iqr,min,max
0,income,396,0.000000,2.193114e+08
1,asinh_income,30,0.000000,1.989915e+01
2,networth,443,-207050.000000,3.077095e+09
3,asinh_networth,229,-12.933863,2.254040e+01
4,liq,364,0.000000,1.478910e+08
5,asinh_liq,0,0.000000,1.950513e+01


In [22]:
print(TAB_DIR.resolve())
print(os.listdir(TAB_DIR))

/content/drive/MyDrive/Portfolio_pilot/data/raw/outputs/tables
['scf_outliers_train.csv']


## Step 9 - Choose the number of archetypes

In [23]:
def make_model(method, k, seed):
    if method == "kmeans":
        return KMeans(n_clusters=k, n_init=20, random_state=seed)
    return GaussianMixture(n_components=k, covariance_type="full", n_init=5, random_state=seed)

tr = feats["train"]
base = weighted_resample(tr, SEED)
scaler_c = StandardScaler().fit(base[CLUSTER_FEATURES])
Xb = scaler_c.transform(base[CLUSTER_FEATURES])
Xt = scaler_c.transform(tr[CLUSTER_FEATURES])
w = tr["weight"].to_numpy(float)
sub = np.random.default_rng(SEED).choice(len(Xb), size=min(3000, len(Xb)), replace=False)

results = []
for method in ["kmeans", "gmm"]:
    for k in K_RANGE:
        m = make_model(method, k, SEED).fit(Xb)
        ref = m.predict(Xt)
        aris = []
        for b in range(N_BOOTSTRAP):
            rs = weighted_resample(tr, SEED + 1 + b)
            mb = make_model(method, k, SEED + 1 + b).fit(scaler_c.transform(rs[CLUSTER_FEATURES]))
            aris.append(adjusted_rand_score(ref, mb.predict(Xt)))
        results.append({
            "method": method, "k": k,
            "silhouette": silhouette_score(Xb[sub], m.predict(Xb)[sub]),
            "bic": m.bic(Xb) if method == "gmm" else np.nan,
            "stability_ari_mean": np.mean(aris), "stability_ari_sd": np.std(aris),
            "min_cluster_share": (np.bincount(ref, weights=w, minlength=k) / w.sum()).min(),
        })
        print("done:", method, k)
ktab = pd.DataFrame(results)
ktab.to_csv(TAB_DIR / "k_selection.csv", index=False)
note("target", "k_selection", ktab.round(4).to_dict(orient="records"))
ktab.round(3)

done: kmeans 2
done: kmeans 3
done: kmeans 4
done: kmeans 5
done: kmeans 6
done: gmm 2
done: gmm 3
done: gmm 4
done: gmm 5
done: gmm 6


,method,k,silhouette,bic,stability_ari_mean,stability_ari_sd,min_cluster_share
0,kmeans,2,0.220,NaN,0.654,0.157,0.399
1,kmeans,3,0.246,NaN,0.733,0.245,0.086
2,kmeans,4,0.267,NaN,0.787,0.171,0.086
3,kmeans,5,0.259,NaN,0.942,0.017,0.086
4,kmeans,6,0.264,NaN,0.646,0.083,0.084
5,gmm,2,0.175,42482.760,0.804,0.108,0.447
6,gmm,3,0.140,39341.450,0.218,0.123,0.204
7,gmm,4,0.204,8286.330,0.875,0.087,0.039
8,gmm,5,0.220,6778.926,0.800,0.193,0.024
9,gmm,6,0.167,-6567.975,0.588,0.120,0.029


In [25]:
print(N_BOOTSTRAP)

20


## Step 10 - Fit the chosen clustering and label every household

In [26]:
SIL_TOLERANCE = 0.01   # silhouette differences smaller than this count as a tie

if CHOSEN_K:
    k = int(CHOSEN_K)
else:
    cand = ktab[(ktab["method"] == CLUSTER_METHOD) & (ktab["min_cluster_share"] >= MIN_CLUSTER_SHARE)]
    if cand.empty:
        cand = ktab[ktab["method"] == CLUSTER_METHOD]
    best_sil = cand["silhouette"].max()
    near_best = cand[cand["silhouette"] >= best_sil - SIL_TOLERANCE]      # options tied with the best
    k = int(near_best.sort_values("stability_ari_mean", ascending=False).iloc[0]["k"])
    print("Tied on silhouette:", near_best["k"].tolist(), "-> most stable:", k)
note("target", "chosen", {"method": CLUSTER_METHOD, "k": k})
print("Chosen:", CLUSTER_METHOD, "k =", k)

final_model = make_model(CLUSTER_METHOD, k, SEED).fit(Xb)

# Order clusters: A0 = most conservative ... A(k-1) = most growth-oriented
lab_b = final_model.predict(Xb)
cols = [CLUSTER_FEATURES.index("risk_willingness"), CLUSTER_FEATURES.index("asinh_networth")]
score = [Xb[lab_b == c][:, cols].sum(axis=1).mean() for c in range(k)]
order = {int(old): new for new, old in enumerate(np.argsort(score))}

for name, d in feats.items():   # fitted on train only; val/test are just labelled
    d["archetype"] = [order[c] for c in final_model.predict(scaler_c.transform(d[CLUSTER_FEATURES]))]

tr = feats["train"]
note("target", "train_weighted_share", (tr.groupby("archetype")["weight"].sum() / tr["weight"].sum()).to_dict())
note("target", "train_count", tr["archetype"].value_counts().sort_index().to_dict())

# Table 3: weighted medians by archetype
prof = tr.groupby("archetype").apply(lambda d: pd.Series({
    **{f"median_{c}": weighted_median(d[c], d["weight"])
       for c in ["age", "income", "networth", "liq", "dti", "liq_to_income", "equity_share"]},
    "mean_risk_willingness": np.average(d["risk_willingness"], weights=d["weight"]),
    "weighted_share": d["weight"].sum() / tr["weight"].sum(),
    "n_households": len(d)}), include_groups=False)
prof.to_csv(TAB_DIR / "archetype_profile_train.csv")
prof.round(2)

Tied on silhouette: [4, 5, 6] -> most stable: 5
Chosen: kmeans k = 5


,median_age,median_income,median_networth,median_liq,median_dti,median_liq_to_income,median_equity_share,mean_risk_willingness,weighted_share,n_households
archetype,,,,,,,,,,
0,39.0,49297.30,-14149.0,1000.0,0.69,0.02,0.00,0.49,0.09,235.0
1,60.0,44162.16,100700.0,1900.0,0.13,0.04,0.00,0.00,0.29,732.0
2,70.0,109891.89,1272300.0,170000.0,0.00,1.41,0.27,0.82,0.09,341.0
3,50.0,145837.84,677750.0,17000.0,1.17,0.11,0.74,0.93,0.25,954.0
4,45.0,93459.46,184650.0,10000.0,0.41,0.10,0.03,1.06,0.28,790.0
